## Assignment 1 in Machine Learning. Done by Kacper, Julianne and Fillip.

In [ ]:
#all imports used in this notebook
import sys
from pathlib import Path
import numpy as np
import seaborn as sns

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path.resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from plotting import plot_histogram
from config import PATH_AMESHOUSING

pd.set_option("display.max_rows", 100)

Step 1a, presenting the data

In [ ]:
df = pd.read_csv(PATH_AMESHOUSING)
display(df.info())
# all present columns and their datatypes as well as their non null counts.

Step 1b, saleprice distribution

In [ ]:

sale_price = df["SalePrice"]
#rounds the values to 2 decimal places and creates a summary of the sale prices

plt.figure(figsize=(14, 10))
sns.histplot(sale_price, bins=100, edgecolor="black", alpha=1)
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.title("Distribution of Sale Prices")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

median = sale_price.median()

plt.figure(figsize=(12, 4))
sns.boxplot(x=sale_price, width=0.15, saturation=1, flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25},
    boxprops={"edgecolor": "black"})
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.text(
    median, 0.2,
    f"Median: ${median:,.0f}",
    ha="center"
)
plt.title("Sale Price", fontweight="bold")
plt.xlabel("Sale Price (USD)")
plt.show()

step 1c, presenting missing values and

In [ ]:
NanValues = df.isnull().sum()
NanValues = NanValues[NanValues > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(NanValues.index, NanValues.values)

plt.yscale('log')
plt.xticks(rotation=45)
plt.ylabel("Number of missing values (log scale)")
plt.xlabel("Column")
plt.title("Missing Values per Column")

# Add the actual count on top of each bar
for bar, value in zip(bars, NanValues.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              str(value), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

Part 1d unusual values, patterns and relationships that might be useful for predicting house prices


In [ ]:
#firstly, here is a correlation map for the int variables compared to sale price: 
target_to_compare_to = df.corr(numeric_only = True)["SalePrice"].sort_values(ascending = False)

plt.figure(figsize = (3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap = "coolwarm", annot = True, cbar = True)

In [ ]:
#looking at some of the highest correlated variables to sale price, 
# we can see that the living area, overall quality, year built, and garage area are 
# all highly correlated with sale price. Let's visualize these relationships with scatter plots and box plots.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="Gr Liv Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(
    data=df,
    x="Overall Qual",
    y="SalePrice",
    width=0.5,
    saturation=1,
    palette="viridis",
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"},
    ax=axes[0, 1]
)
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(
    data=df,
    x="Year Built",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(
    data=df,
    x="Garage Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(
        mtick.StrMethodFormatter('${x:,.0f}')
    )
    ax.title.set_fontweight("bold")
    legend = ax.get_legend()
    if legend:
        legend.set_title("Overall Qual", prop={"size": 10})
        for text in legend.get_texts():
            text.set_fontsize(8)

plt.tight_layout()
plt.show()

In [ ]:
#Lets now look at the distribution of sale prices by overall quality, the same plot as earlier. 
#This will help us understand how the overall quality of a house affects its sale price.
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df,
    x="SalePrice",
    hue="Overall Qual",
    bins=100,
    multiple="stack",
    edgecolor="black",
    alpha=1,
    palette="viridis"
)

plt.gca().xaxis.set_major_formatter(
    mtick.StrMethodFormatter("${x:,.0f}")
)

plt.title("Distribution of Sale Prices by Overall Quality", fontweight="bold")
plt.grid(True)
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

#this table also shows the amount of the different ratings for the houses. Since 10s, 1s, 2s and 3s 
# are rarer they might spike the graph more than other quality levels.
quality_counts = (
    df["Overall Qual"]
    .value_counts()
    .sort_index()
    .rename_axis("Overall Qual")
    .reset_index(name="Amount")
)

quality_table = quality_counts.rename(
    columns={
        "Overall Qual": "Overall Quality",
        "Amount": "Number of Houses"
    }
)

display(quality_table)

From personal experience, they neighbourhood a house is in creates an artificial boost or lowering in price, lets explroe that.

In [ ]:
order_by_price = (
    df.groupby("Neighborhood")["SalePrice"]
      .median()
      .sort_values()
      .index
)

order_by_name = sorted(
    df["Neighborhood"]
    .dropna()
    .unique()
)


# Sorted by sale price
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=order_by_price,
    hue="Neighborhood",
    palette="Spectral",
    legend=False,
    width=0.6,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.xticks(rotation=65, ha="right")
plt.title("Sale Price Distributions by Neighbourhood - Sorted by Price",
          fontweight="bold")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()